In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
print("All modules imported successfully!")

In [ ]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self, emb, latent, heads=6, **kwargs):
        super().__init__(**kwargs)
        self.emb = emb
        self.latent = latent
        self.heads = heads

        # 1. Separate projections for Q, K, and V for all heads combined
        self.wq = Dense(self.latent * self.heads)
        self.wk = Dense(self.latent * self.heads)
        self.wv = Dense(self.latent * self.heads)
        
        # 2. Final projection to ensure the output matches 'emb' size for the residual connection
        self.dense_proj = Dense(self.emb)

        self.dense = Sequential([
            Dense(latent, activation='relu'),
            Dense(emb)
        ])
        
        # Initialize Layer Normalization blocks
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)

    def pos_enc(self, duncan_seq_len):
        pos = tf.cast(tf.range(duncan_seq_len), tf.float32)[:, tf.newaxis]
        dimensions = tf.cast(tf.range(self.emb), tf.float32)[tf.newaxis, :]
        angle_rates = 1 / tf.pow(10000.0, (2 * (dimensions // 2)) / tf.cast(self.emb, tf.float32))
        angle_rads = pos * angle_rates
        
        sines = tf.math.sin(angle_rads[:, 0::2])
        cosines = tf.math.cos(angle_rads[:, 1::2])
        
        pos_encoding = tf.stack([sines, cosines], axis=-1)
        pos_encoding = tf.reshape(pos_encoding, [duncan_seq_len, self.emb])
        return pos_encoding

    def split_heads(self, x, batch_size):
        """
        Splits the last dimension into (heads, latent).
        Transposes the result so the shape is (batch_size, heads, seq_len, latent)
        """
        x = tf.reshape(x, (batch_size, -1, self.heads, self.latent))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        """
        inputs: (batch_size, duncan_seq_len, emb)
        """
        bs = tf.shape(inputs)[0] # Dynamic batch size
        sl = tf.shape(inputs)[1] # Dynamic sequence length
        
        # 1. Add Positional Encoding
        pos_enc = self.pos_enc(sl) 
        x = inputs + pos_enc # (batch_size, seq_len, emb)

        # 2. Linear Projections
        q = self.wq(x)  # (batch_size, seq_len, latent * heads)
        k = self.wk(x)  # (batch_size, seq_len, latent * heads)
        v = self.wv(x)  # (batch_size, seq_len, latent * heads)

        # 3. Split into Multi-Heads
        q = self.split_heads(q, bs) # (batch_size, heads, seq_len, latent)
        k = self.split_heads(k, bs) # (batch_size, heads, seq_len, latent)
        v = self.split_heads(v, bs) # (batch_size, heads, seq_len, latent)

        # 4. Scaled Dot-Product Attention
        # transpose_b=True swaps the last two dimensions of K -> (batch_size, heads, latent, seq_len)
        attention_scores = tf.matmul(q, k, transpose_b=True) # (batch_size, heads, seq_len, seq_len)
        attention_scores = attention_scores / tf.math.sqrt(tf.cast(self.latent, tf.float32))
        
        attention_weights = tf.nn.softmax(attention_scores, axis=-1)
        
        # Multiply weights by V
        attention_output = tf.matmul(attention_weights, v) # (batch_size, heads, seq_len, latent)

        # 5. Recombine Heads
        # Transpose back to (batch_size, seq_len, heads, latent)
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        # Flatten the last two dimensions
        concat_attention = tf.reshape(attention_output, (bs, sl, self.latent * self.heads)) # (batch_size, seq_len, latent * heads)

        # 6. Final Linear Projection and First Residual
        proj_attention = self.dense_proj(concat_attention) # (batch_size, seq_len, emb)
        out1 = self.layernorm1(proj_attention + x) # Residual connection now works perfectly

        # 7. Feed-Forward Network and Second Residual
        ffn_output = self.dense(out1) # (batch_size, seq_len, emb)
        encoder_output = self.layernorm2(ffn_output + out1) # (batch_size, seq_len, emb)

        return encoder_output